In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import random
import scipy

In [ ]:
def gremban_lift(L):
    """
    Constructs the Gremban expansion matrix M from a given matrix L.
    
    Args:
        L (numpy.ndarray): The input matrix (usually a Laplacian).
        
    Returns:
        numpy.ndarray: The 2n x 2n Gremban expansion matrix.
    """
    # Ensure input is a NumPy array
    L = np.array(L)
    
    # Get the size of the matrix (number of rows)
    n = L.shape[0]
    
    # Initialize a 2n x 2n matrix of zeros
    M = np.zeros((2 * n, 2 * n), dtype=float)

    for i in range(n):
        # Set diagonal elements for both diagonal blocks
        # Julia: M[i,i] and M[i+n,i+n]
        
        #M[i, i] = L[i, i]
        #M[i + n, i + n] = L[i, i]

        # Loop through the upper triangle (excluding diagonal)
        # Julia: for j in i+1:n
        Off_set_diag= 0
        for j in range(n):
            if j==i: 
                continue
            val = L[i, j]
            
            if val > 0:
                # "Bad" edges (positive off-diagonals) move to the off-diagonal blocks (A)
                # We negate the value as per the standard Gremban reduction
                M[i, n + j] = -val
                M[j, n + i] = -val
                M[n + i, j] = -val
                M[n + j, i] = -val
                Off_set_diag += val
                #print(f" sum is {Off_set_diag}, val is {val}")
            else:
                # "Good" edges (negative off-diagonals) stay in the diagonal blocks
                M[i, j] = val
                M[j, i] = val
                M[i + n, j + n] = val
                M[j + n, i + n] = val
                Off_set_diag += -val
                print(f" sum is{Off_set_diag}, val is {-val}")
        #print(f" round {i}, off is {Off_set_diag}")
        diff = L[i,i]-Off_set_diag
        M[i,i] = Off_set_diag+ diff/2
        M[i + n, i + n] = M[i, i]    
        M[i,i+n] =    -diff/2
        M[i+n,i] = -diff/2
    return M
    

In [ ]:
def gremban_lift_hermitian(L,N_max):
    """
    Constructs the Gremban expansion matrix M from a given matrix L.
    
    Args:
        L (numpy.ndarray): The input matrix (usually a Laplacian).
        N_max: Size of the lift, depends on the 
        
    Returns:
        numpy.ndarray: The 2n x 2n Gremban expansion matrix.
    """
    # Ensure input is a NumPy array
    L = np.array(L)
    
    # Get the size of the matrix (number of rows)
    n = L.shape[0]
    
    # Initialize a 2n x 2n matrix of zeros
    M = np.zeros((N_max * n, N_max * n), dtype=float)

    for i in range(n):
        # Set diagonal elements for both diagonal blocks
        # Julia: M[i,i] and M[i+n,i+n]
        
        #M[i, i] = L[i, i]
        #M[i + n, i + n] = L[i, i]

        # Loop through the upper triangle (excluding diagonal)
        # Julia: for j in i+1:n
        Off_set_diag= 0
        for j in range(n):
            if j==i: 
                continue
            val = -L[i, j]
            for k in range(N_max):
               # print(np.exp(1j*2 * np.pi * k/N_max)*val)
               #if (k==3 and i==1 and j==3) or (k==18 and i==1 and j==3) : 
                #print(f"current multiple is {np.exp(1j*2 * np.pi * k/N_max)*val}")
                #print(f"the current gap is {np.abs(val)-np.exp(1j*2 * np.pi * k/N_max)*val} and k is {3}")
               if np.isclose(np.exp(-1j*2 * np.pi * k/N_max)*val,np.abs(val),atol=1e-4):
                    Off_set_diag += np.abs(val) 
                    for l in range(N_max): 
                     #   print(f"block index {k+l%N_max} and small index {j}")
                        M[n*l+i,n*((k+l)%N_max)+j]= -np.abs(val)
                        
        #print(f" round {i}, off is {Off_set_diag}")
        diff = L[i,i]-Off_set_diag
        for k in range(N_max):
            #print(f"accessing {n*k} with current value {M[n*k+i,n*k+i]}")
            M[n*k+i,n*k+i]=Off_set_diag+ diff/2   
            #print(f"changed {n*k+i} with current value {M[n*k+i,n*k+i]}")
            #print(f"block index {((int(N_max/2)+k)%N_max)} and small index {i} and runner {k} and step size {int(N_max/2)}")
            M[i+k*n,i+((int(N_max/2)+k)%N_max) *n] =    -diff/2
    return M

In [ ]:
def transform_to_sparse(L, N_max):
    """
    Efficiently transforms matrix L into M using sparse storage.
    L: Input matrix (can be dense or sparse)
    N_max: Expansion factor
    """
    # Convert L to COO format to efficiently iterate over non-zero edges
    L_coo = sparse.coo_matrix(L)
    n = L_coo.shape[0]
    
    # Lists to store the coordinates and data for the new sparse matrix M
    m_rows = []
    m_cols = []
    m_data = []
    
    # To track the diagonal offset for each node i
    off_set_diag = np.zeros(n)
    
    # 1. Process Off-Diagonal Elements (Edges)
    # Filter for i != j
    mask = L_coo.row != L_coo.col
    rows, cols, vals = L_coo.row[mask], L_coo.col[mask], L_coo.data[mask]
    
    for i, j, val in zip(rows, cols, vals):
        v_neg = -val
        abs_v = np.abs(v_neg)
        
        # Calculate the phase k that satisfies the condition
        # exp(1j * 2pi * k / N_max) = abs(v) / (-L[i,j])
        target_phase = abs_v / v_neg
        angle = np.angle(target_phase)
        if angle < 0: angle += 2 * np.pi
        
        # Find the integer k that best matches this phase
        k = int(round(angle * N_max / (2 * np.pi))) % N_max
        
        # Accumulate offset
        off_set_diag[i] += abs_v
        
        # Vectorized expansion for all l layers
        l_range = np.arange(N_max)
        m_rows.extend(n * l_range + i)
        m_cols.extend(n * ((k + l_range) % N_max) + j)
        m_data.extend([-abs_v] * N_max)

    # 2. Process Diagonals
    L_diag = L_coo.diagonal()
    shift = int(N_max // 2)
    
    for i in range(n):
        diff = L_diag[i] - off_set_diag[i]
        k_range = np.arange(N_max)
        
        # Update Main Diagonal blocks
        m_rows.extend(n * k_range + i)
        m_cols.extend(n * k_range + i)
        m_data.extend([off_set_diag[i] + diff / 2] * N_max)
        
        # Update Mirrored Diagonal blocks
        m_rows.extend(n * k_range + i)
        m_cols.extend(n * ((k_range + shift) % N_max) + i)
        m_data.extend([-diff / 2] * N_max)
        
    # Construct the final sparse matrix (CSR format is efficient for math)
    M = sparse.coo_matrix((m_data, (m_rows, m_cols)), 
                          shape=(N_max * n, N_max * n))
    return M.tocsr()

In [ ]:

def vectorized_transform(L, N_max):
    L = np.array(L)
    n = L.shape[0]
    
    # Pre-calculate the phase factors for all k
    # Shape: (N_max,)
    k_indices = np.arange(N_max)
    phases = np.exp(1j * -2 * np.pi * k_indices / N_max)
    
    # Initialize M
    M = np.zeros((N_max * n, N_max * n), dtype=float)
    
    # We still iterate over i, j to handle the graph edges
    # But we vectorize the k and l logic
    off_set_diags = np.zeros(n)
    
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            
            val = -L[i, j]
            if val == 0:
                continue
                
            # Vectorized check: which k satisfies the phase condition?
            # Using np.isclose across all k simultaneously
            abs_val = np.abs(val)
            condition = np.isclose(phases * val, abs_val, atol=1e-6)
            
            if np.any(condition):
                # Get the specific k that matched (assuming only one k matches)
                k_match = np.where(condition)[0][0]
                off_set_diags[i] += abs_val
                
                # Vectorized update for all l
                # Target row indices: n*l + i
                # Target col indices: n*((k_match + l) % N_max) + j
                l_indices = np.arange(N_max)
                rows = n * l_indices + i
                cols = n * ((k_match + l_indices) % N_max) + j
                M[rows, cols] = -abs_val
        
        # Apply the diagonal and "mirrored" layer logic for node i
        diff = L[i, i] - off_set_diags[i]
        k_range = np.arange(N_max)
        
        # Main diagonal update
        diag_idx = n * k_range + i
        M[diag_idx, diag_idx] = off_set_diags[i] + diff / 2
        
        # Mirrored layer update (Shifted by N_max/2)
        shift = int(N_max / 2)
        mirror_cols = n * ((k_range + shift) % N_max) + i
        M[diag_idx, mirror_cols] = -diff / 2
        
    return M

In [ ]:
def deconstruct_l_matr(l_to_decon,N,a,b,k):
    l_size= l_to_decon.shape[0]
    l_tilde_size= int(l_size*N)
    matrix_deconstruct= np.zeros([l_tilde_size,N],dtype=complex)
    for i in range(N):
        b_i= np.zeros(l_tilde_size,dtype=complex)
        b_i[i*l_size+a]= 1
        #print(f"index {i} postion{(i+k)%N*l_size}")
        b_i[(i+k)%N*l_size+b]=-1
        matrix_deconstruct[:,i] = b_i
    return matrix_deconstruct

In [ ]:
def eff_ref_and_trans_imb_matr(lift,l_new,N_max,a,b,phi_k):
    pinv_lift= np.linalg.pinv(lift)
    b_matrix= deconstruct_l_matr(l_new,N_max,a,b,phi_k)
    print(b_matrix.shape)
    zero_total =0
    total_eff_ref=0
    trans_tot_abs=0
    r_eff_matrix= np.transpose(b_matrix)@pinv_lift@ b_matrix
    #print(r_eff_matrix[0,1])
    for i in range(b_matrix.shape[1]):
        #print(np.sum(np.abs(r_eff_matrix[:,i]))-r_eff_matrix[i,i])
        for j in range(i):
            trans_imb= r_eff_matrix[i,j]
            trans_tot_abs+= 2* np.abs(trans_imb)
            zero_total+= 2 * np.cos(2*np.pi *(j-i)/N_max)* trans_imb
    #print(r_eff_matrix[0,0])
    total_eff_ref= np.sum(r_eff_matrix.diagonal()) 
    #print(f"total sum is  {np.sum(np.abs(r_eff_matrix))}")      
    print(f"zero total impedenance is  {zero_total}")
    #print(f"total trans imbedance absolute is {trans_tot_abs}")
    print(f"total eff ref is {total_eff_ref}")
    return total_eff_ref,zero_total

In [ ]:
a=1
b=2
r_a= -4652
r_b= 2102
l_new=np.array([1,0,0,-1])
N_max= np.lcm(a,b)
test_SDD = [
    [ np.abs(r_a)+2, 0,  r_a,-2],
    [0,   np.abs(r_b)+1, -1,r_b],
    [r_a, -1, np.abs(r_a)+4,-3],
    [-2,r_b,-3,np.abs(r_b)+5]
]
print(f"i do change again")
lift_test= gremban_lift_hermitian(test_SDD,N_max)
lift_test_ssd= gremban_lift(test_SDD)
lift_test_vect= vectorized_transform(test_SDD,N_max)
new_tot_reff, new_total_trans_imb= eff_ref_and_trans_imb_matr(lift_test,l_new,2,0,3,0)
tot_sdd,trans_imb_sdd= eff_ref_and_trans_imb_matr(lift_test_ssd,l_new,2,0,3,0)
tot_sdd,trans_imb_sdd= eff_ref_and_trans_imb_matr(lift_test_vect,l_new,2,0,3,0)

In [ ]:

#-830.1062348866983 -1066.6990044849356 9 6 [1.+1.2246468e-16j 0.+0.0000000e+00j 0.+0.0000000e+00j 1.+0.0000000e+00j]
#-830.1062348866983 -1066.6990044849356 9 6 [1.+1.2246468e-16j 0.+0.0000000e+00j 0.+0.0000000e+00j 1.+0.0000000e+00j]
a= 17*17
b=17*2
r_a= -8300
r_b= -1066
N_max= np.lcm(a,b)
print(N_max)
#l_new=np.array([-np.exp(1j*2*np.pi/N_max),0,0,1])
l_new=np.array([-np.exp(1j*2*np.pi*a/N_max),0,0,1])
l_1= np.array([0,-1,0,1])
print(l_new)
test_SDD = [
    [ np.abs(r_a)+2, 0,  r_a*np.exp(1*1j*2*np.pi/a),-2],
    [0,   np.abs(r_b)+1, -1,r_b*np.exp(-1*1j*2*np.pi/b)],
    [r_a* np.exp(-1j*2*np.pi/a), -1, np.abs(r_a)+4,-3],
    [-2,r_b*np.exp(1j*2*np.pi/b),-3,np.abs(r_b)+5]
]

lift_test= gremban_lift_hermitian(test_SDD,N_max)
lift_test_vect= vectorized_transform(test_SDD,N_max)
#new_tot_reff, new_total_trans_imb= eff_ref_and_trans_imb_matr(lift_test,l_new,N_max,3,0,0)
new_tot_reff, new_total_trans_imb= eff_ref_and_trans_imb_matr(lift_test_vect,l_new,N_max,3,0,a)
print(eff_ref_and_trans_imb_matr(lift_test_vect,l_1,N_max,3,1,0))


In [ ]:
for i in range(100): 
    r_a = (np.random.rand() - 0.5) * 10000  # scalar
    r_b = (np.random.rand() - 0.5) * 10000  # scalar
    phi_a = random.randint(1,30)
    phi_b = random.randint(1,30)
    N_max= np.lcm(2*phi_a,phi_b)
    l_new=np.array([-np.exp(-1*1j*2*np.pi*phi_a/N_max),0,0,1])
    l_1= np.array([0,-np.exp(-1*1j*2*np.pi*phi_b/N_max),0,1])
    l_2= np.array([0,-np.exp(-1*1j*2*np.pi*phi_b/N_max),1,0])
    l_3= np.array([0,0,-1,1])
    test_SDD_rand = [
    [ 2+np.abs(r_a), 0,  r_a*np.exp(-1*1j*2*np.pi/phi_a),-2],
    [0,  1+np.abs(r_b), -1,r_b*np.exp(-1*1j*2*np.pi/phi_b)],
    [r_a * np.exp(1j*2*np.pi/phi_a), -1, 4+np.abs(r_a),-3],
    [-2,r_b*np.exp(1j*2*np.pi/phi_b),-3,5+np.abs(r_b)]
    ]
    lift_rand= vectorized_transform(test_SDD_rand,N_max)
    new_tot_reff, new_total_trans_imb= eff_ref_and_trans_imb_matr(lift_rand,l_new,N_max,3,0,phi_a)
    print(new_tot_reff,new_total_trans_imb,phi_a,phi_b,r_a,r_b,a,b)
    if np.abs(new_total_trans_imb)>=new_tot_reff:
        print(r_a,r_b,phi_a,phi_b,l_new)
        break
    new_tot_reff, new_total_trans_imb= eff_ref_and_trans_imb_matr(lift_rand,l_1,N_max,3,1,phi_b)
    print(new_tot_reff,new_total_trans_imb,phi_a,phi_b,r_a,r_b,a,b)
    if np.abs(new_total_trans_imb)>=new_tot_reff:
        print(r_a,r_b,phi_a,phi_b,l_1)
        break
    new_tot_reff, new_total_trans_imb= eff_ref_and_trans_imb_matr(lift_rand,l_2,N_max,2,1,phi_b)
    print(new_tot_reff,new_total_trans_imb,phi_a,phi_b,r_a,r_b,a,b)
    if np.abs(new_total_trans_imb)>=new_tot_reff:
        print(r_a,r_b,phi_a,phi_b,l_2)
        break
    new_tot_reff, new_total_trans_imb= eff_ref_and_trans_imb_matr(lift_rand,l_2,N_max,3,2,0)
    print(new_tot_reff,new_total_trans_imb,phi_a,phi_b,r_a,r_b,a,b)
    if np.abs(new_total_trans_imb)>=new_tot_reff:
        print(r_a,r_b,phi_a,phi_b,l_3)
        break





In [ ]:
exmp_SDD = [
    [ 11, 10,0,0,-1],
    [10,12, 0,-1,-1],
    [0,0 , 2,-1,-1],
    [0,-1,-1,4,2],
    [-1,-1,-1,2,5]
]
l_new=np.array([1,-1,0,0,0])
l_1= np.array([0,-1,1,0,0])
print(l_new)
N_max=2
lift_test= gremban_lift_hermitian(exmp_SDD,N_max)
lift_test_vect= vectorized_transform(exmp_SDD,N_max)
print(lift_test_vect.shape)
#new_tot_reff, new_total_trans_imb= eff_ref_and_trans_imb_matr(lift_test,l_new,N_max,3,0,0)
new_tot_reff, new_total_trans_imb= eff_ref_and_trans_imb_matr(lift_test_vect,l_new,2,1,0,0)
total_new_edge= new_tot_reff+new_total_trans_imb
#print(eff_ref_and_trans_imb_matr(lift_test_vect,l_1,N_max,2,1,0))
tot_reff_one, total_trans_imb_one= eff_ref_and_trans_imb_matr(lift_test_vect,np.array([1,0,0,0,-1]),N_max,4,0,0)
total_edge_one= tot_reff_one+total_trans_imb_one
tot_reff_two, total_trans_imb_two= eff_ref_and_trans_imb_matr(lift_test_vect,np.array([0,1,0,0,-1]),N_max,4,1,0)
total_edge_two= tot_reff_two+total_trans_imb_two
print(total_new_edge,total_edge_one+total_edge_two)
print(f"vector {np.array([1,1,0,0])} result  {eff_ref_and_trans_imb_matr(lift_test_vect,np.array([1,1,0,0,0]),N_max,1,0,1)}")
print(f"vector {np.array([-1,0,1,0])} result  {eff_ref_and_trans_imb_matr(lift_test_vect,np.array([-1,0,1,0,0]),N_max,2,0,0)}")
print()

In [ ]:
np.array([1,0,0,0,-1]).T@ np.linalg.pinv(exmp_SDD)@ np.array([0,1,0,0,-1])

In [ ]:
hr_eff =l_new @np.linalg.pinv(exmp_SDD)@l_new.conj().T
hr_eff
2*hr_eff-new_tot_reff-new_total_trans_imb